# 05 · Answer Generation — `scale_1K`
*Generates answers via OpenAI and saves `eval_runs.parquet`. Run 05b for RAGAS evaluation.*

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_1K"
N_PAPERS      = 1000
RANDOM_SEED   = 42
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 50
RRF_K         = 60
TOP_K_STAGE1  = 50

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)

BASE_DIR    = os.path.abspath("../..")
RESULTS_DIR = os.path.join(BASE_DIR, "4_results", SCALE_LABEL)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Scale: {SCALE_LABEL} | Device: {DEVICE} | Results: {RESULTS_DIR}")

Scale: scale_1K | Device: cuda | Results: d:\SciRet-Scientific-Information-Made-Easy\Sciret2\4_results\scale_1K


In [2]:
from dotenv import load_dotenv
load_dotenv(os.path.join(BASE_DIR, ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
print("OpenAI key loaded ✓")

GEN_MODEL = "gpt-4o-mini"
print(f"Generation model: {GEN_MODEL}")

OpenAI key loaded ✓
Generation model: gpt-4o-mini


In [3]:
# Quick API test
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
resp = client.chat.completions.create(
    model=GEN_MODEL,
    messages=[{"role": "user", "content": "Say hello in one word."}],
    max_tokens=10,
)
print("API test OK:", resp.choices[0].message.content)

API test OK: Hello!


In [4]:
import pickle
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

df_chunks  = pd.read_parquet(f"{RESULTS_DIR}/chunks.parquet")
texts      = df_chunks["text"].tolist()
chunk_ids  = df_chunks["chunk_id"].tolist()
id_to_text = dict(zip(chunk_ids, texts))

embeddings = np.load(f"{RESULTS_DIR}/bge_m3_embeddings.npy")
model      = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

with open(f"{RESULTS_DIR}/bm25_index.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open(os.path.join(BASE_DIR, "1_data/eval/queries.json")) as f:
    QUERIES = json.load(f)

print(f"Queries: {len(QUERIES)} | Chunks: {len(texts):,}")

d:\SciRet-Scientific-Information-Made-Easy\Sciret2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 25066.46it/s]


Queries: 50 | Chunks: 1,029


In [5]:
def generate_answer(query, context_chunks):
    context = "\n\n".join([f"[{i+1}] {c}" for i, c in enumerate(context_chunks)])
    prompt  = (
        "You are a scientific literature assistant. "
        "Answer the question using ONLY the provided context. "
        "Cite source numbers [1], [2], etc. for each claim.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    )
    try:
        resp = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=2048,
            temperature=0.1,
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"ERROR: {e}"

def hybrid_retrieve_top5(query):
    q_emb     = model.encode([query], normalize_embeddings=True)
    sims      = cosine_similarity(q_emb, embeddings)[0]
    dense_top = list(np.argsort(sims)[::-1][:50])
    tokens    = query.lower().split()
    bm25_top  = list(np.argsort(bm25.get_scores(tokens))[::-1][:50])
    rrf = {}
    for lst in [dense_top, bm25_top]:
        for rank, idx in enumerate(lst):
            cid = chunk_ids[idx]
            rrf[cid] = rrf.get(cid, 0) + 1.0 / (RRF_K + rank + 1)
    top5 = sorted(rrf, key=rrf.get, reverse=True)[:5]
    return [id_to_text[c] for c in top5 if c in id_to_text]

print("Generating answers...")
eval_records = []
for i, query in enumerate(QUERIES):
    contexts = hybrid_retrieve_top5(query)
    answer   = generate_answer(query, contexts)
    eval_records.append({"question": query, "answer": answer,
                         "contexts": contexts, "ground_truth": query})
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(QUERIES)} done")

pd.DataFrame(eval_records).to_parquet(f"{RESULTS_DIR}/eval_runs.parquet", index=False)
print(f"\nSaved {len(eval_records)} answers → {RESULTS_DIR}/eval_runs.parquet")

errors = sum(1 for r in eval_records if str(r['answer']).startswith('ERROR:'))
print(f"Errors: {errors}/{len(eval_records)} — run 05b_ragas_eval.ipynb next")

Generating answers...
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done

Saved 50 answers → d:\SciRet-Scientific-Information-Made-Easy\Sciret2\4_results\scale_1K/eval_runs.parquet
Errors: 0/50 — run 05b_ragas_eval.ipynb next
